# Single Store Team — Kaggle CPU
Accelerator=None, Internet=On. Use a fresh session. This notebook starts MySQL/API locally, never your personal database.
Commit/push the tested source first. Set `REF` (pushed SHA or branch) and `SCENARIO` (`stress`, `hotspot` or `soak`) in the next cell.

In [ ]:
from pathlib import Path
import subprocess
import sys

REF = 'son'  # Pushed commit SHA (preferred) or branch to benchmark
SCENARIO = 'hotspot'  # 'stress' (TEST-00), 'hotspot' (TEST-00b) or 'soak' (30 minutes)

repo = Path('/kaggle/working/store_ktpm')
if repo.exists():
    raise RuntimeError('Repository already exists; use a fresh Kaggle session')
url = 'https://github.com/HuuHieu03/store_ktpm_phase2_group14.git'
subprocess.run(['git', 'clone', url, str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', REF], check=True)
if not (repo / 'performance/kaggle_run.py').is_file():
    raise RuntimeError('This commit does not include the load runner')
print('Commit:', subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip())
print('Scenario:', SCENARIO)


In [ ]:
import os
env = os.environ.copy()
env['LOAD_SCENARIO'] = SCENARIO
env.pop('LOAD_SMOKE_ONLY', None)
# Feasibility check only (first phase), then restart the session for the real run:
# env['LOAD_SMOKE_ONLY'] = '1'
subprocess.run([sys.executable, str(repo / 'performance/kaggle_run.py')], cwd=repo, env=env, check=True)


In [ ]:
import shutil
runs = sorted(Path('/kaggle/working/single_store_load').glob('*'))
if not runs:
    raise RuntimeError('No evidence directory; inspect the runner error above')
latest = runs[-1]
report = latest / 'REPORT.md'
print(report.read_text(encoding='utf-8') if report.exists() else 'No completed report: inspect api.log/runner output')
archive = shutil.make_archive('/kaggle/working/single_store_load_results', 'zip', root_dir='/kaggle/working', base_dir='single_store_load')
print('Download from Kaggle Output:', archive)
